# STG 3 — Filtro de títulos sin valor semántico

**Objetivo**: eliminar de `df_consolidado` los registros cuyo `titulo_base` no describe
el contenido temático de una ley (mociones, habilitaciones, expedientes sin descripción, etc.).

**Entrada**: `data/df_consolidado.csv`  
**Salida**: `data/df_modelado.csv`

> ⚠️ Este notebook tiene dos celdas críticas separadas:
> - **Celda de INSPECCIÓN**: muestra qué se va a filtrar. No elimina nada.
> - **Celda de APLICACIÓN**: aplica el filtro y guarda. Ejecutar solo después de revisar la inspección.

In [ ]:
import pandas as pd
import re

df = pd.read_csv('../data/df_consolidado.csv')

print('Columnas:', df.columns.tolist())
print('Shape:', df.shape)
print('\nPrimeras filas:')
df.head()

## Función de filtrado

Define los patrones que identifican títulos sin valor semántico.
Cada grupo corresponde a una categoría documentada en la spec 002.

In [ ]:
PATRONES_SIN_VALOR = [
    # Habilitación de tratamiento (procedimiento parlamentario, no ley sustantiva)
    r'^HABILITACI[OÓ]N DEL TRATAMIENTO',
    r'^HABILITACION DEL TRATAMIENTO',
    r'^HABILITACI[OÓ]N DE TRATAMIENTO',
    r'^HABILITACI[OÓ]N PROYECTOS DE',
    r'^Habilitar el [Tt]ratamiento del Exp',
    r'^Habilitaci.n de temas',

    # Insistencia de ley (solo número, sin descripción)
    r'^INSISTENCIA PROYECTO DE LEY\s+[\d\.]+\s*$',

    # Moción de emplazamiento
    r'^MOCI[OÓ]N DE EMPLAZAMIENTO',
    r'^MOCION DE EMPLAZAMIENTO',
    r'^Moci[oó]n de emplazamiento',

    # Moción de reconsideración
    r'^MOCION DE RECONSIDERACI[OÓ]N',
    r'^MOCI[OÓ]N DE RECONSIDERACI[OÓ]N',
    r'^Moci[oó]n de reconsideraci[oó]n',

    # Moción de orden
    r'^Moci[oó]n de Orden',
    r'^MOCI[OÓ]N DE ORDEN',

    # Moción del/de la Diputado/a
    r'^Moci[oó]n de[l]?\s+(la\s+)?(Diputad[ao]|vuelta)',
    r'^MOCI[OÓ]N DE[L]?\s+(LA\s+)?(DIPUTAD[AO]|VUELTA)',

    # Moción genérica
    r'^MOCION SOLICITADA',
    r'^mocion solicitada',

    # Apartamiento del reglamento
    r'^APARTAMIENTO DEL REGLAMENTO',
    r'^Moci[oó]n de apartamiento del reglamento',

    # Pedido de preferencia / tratamiento sobre tablas
    r'^Pedido de [Pp]referencia',
    r'^Pedido de [Tt]rat(amiento)?\.?\s+sobre\s+[Tt]abla',

    # Solicitud de licencia de diputado
    r'^Solicitud de [Ll]icencia',

    # Renuncia de diputado
    r'^Renuncia del [Dd]ip',

    # Votación de desarrollo de sesión
    r'^VOTACI.N DESARROLLO DE LA SESI',

    # Apertura de sobre de declaración jurada patrimonial
    r'^Proceder a la Apertura del sobre',

    # Órdenes del día sin descripción (sin puntos)
    r'^OD\s+\d+\s*[-–]\s*\d+',
    r'^od\s+\d+\s*[-–]\s*od\s+\d+',

    # Órdenes del día sin descripción (con puntos)
    r'^O\.D\.\s*\d+\s*$',
    r'^O\.D\.\s*\d+\s*[-–,]\s*O\.D\.',

    # Plan de labor
    r'^PLAN DE LABOR$',

    # Votación en general sin título propio
    r'^VOTACI[OÓ]N EN GRAL\.',

    # Pliegos de magistrados de la Corte Suprema
    r'^Exp\.\s+7835-D-00',
]


def _tiene_descripcion(titulo):
    """
    Devuelve True si hay texto descriptivo después del número de expediente.

    Strips en cadena:
      1. Prefijo Exp./Expte./Exped./Expediente/Expedientes
      2. Número de expediente (NNNN-XX-NNNN o XX-NNNN)
      3a. "(y otro/s)" con paréntesis
      3b. "y otro/s" sin paréntesis
      4. Segundo número de expediente (y NNN-XX-NNN)
      5. Separadores (. - / * espacio)
      6a. "Orden del Día NNN"
      6b. "O.D. NNN" (con punto)
      6b2. "O. del Día NNN" (con "del")
      6c. Re-strip separadores
      6d. "Vot(ación). en General..." / "Votación Artículos NN"
    """
    sin_prefijo = re.sub(r'^Exp(ediente|ed|te)?s?\.?\s*', '', titulo.strip(), flags=re.IGNORECASE)

    sin_numero = re.sub(r'^\d+\s*[-–]\s*[A-Za-z]+\s*[-–]\s*\d+\s*', '', sin_prefijo)
    if sin_numero == sin_prefijo:
        sin_numero = re.sub(r'^[A-Za-z]+-\d+(-\d+)?\s*$', '', sin_prefijo)

    # 3a. "(y otro/s)" con paréntesis
    sin_numero = re.sub(r'^\(y\s+otro[s]?\)\s*', '', sin_numero.strip(), flags=re.IGNORECASE)
    # 3b. "y otro/s" sin paréntesis
    sin_numero = re.sub(r'^y\s+otros?\s*', '', sin_numero.strip(), flags=re.IGNORECASE)

    # 4. Segundo número de expediente
    sin_numero = re.sub(r'^y\s+\d+\s*[-–]\s*[A-Za-z]+\s*[-–]\s*\d+\s*', '', sin_numero.strip(), flags=re.IGNORECASE)

    sin_sep = re.sub(r'^[\s\.\-\/\*]+', '', sin_numero.strip())

    # 6a. "Orden del Día NNN"
    sin_sep = re.sub(r'^Orden del D[íi]a\s+\d+\s*', '', sin_sep.strip(), flags=re.IGNORECASE)
    # 6b. "O.D. NNN" (O punto D punto)
    sin_sep = re.sub(r'^O\.?\s*D\.?\s*\d+\s*', '', sin_sep.strip(), flags=re.IGNORECASE)
    # 6b2. "O. del Día NNN" (forma con "del") — ej: "O. del Día 884"
    sin_sep = re.sub(r'^O\.\s*del\s+D.a\s+\d+\s*', '', sin_sep.strip(), flags=re.IGNORECASE)

    sin_sep = re.sub(r'^[\s\.\-\/\*,]+', '', sin_sep.strip())

    sin_sep = re.sub(r'^Vot(aci.n)?\.?\s+en\s+General.*$', '', sin_sep.strip(), flags=re.IGNORECASE)
    sin_sep = re.sub(r'^Votaci.n Art.culos?\s+\d+.*$', '', sin_sep.strip(), flags=re.IGNORECASE)

    return len(sin_sep.strip()) > 5


def es_sin_valor(titulo):
    """Devuelve True si el título no tiene valor semántico para el modelo."""
    t = str(titulo).strip()

    for patron in PATRONES_SIN_VALOR:
        if re.search(patron, t, re.IGNORECASE):
            return True

    if re.match(r'^Exp(ediente|ed|te)?s?\.?\s*\d', t, re.IGNORECASE):
        if not _tiene_descripcion(t):
            return True

    return False


print('Función es_sin_valor() definida.')

casos_prueba = [
    ('EXPTE. 5297-D-2025',                                                        True),
    ('INSISTENCIA PROYECTO DE LEY 27.795',                                        True),
    ('HABILITACIÓN DEL TRATAMIENTO EXPTE. 9-PE-2025',                             True),
    ('Habilitar el Tratamiento del Expediente 4881-D-2017. De ley. Impuesto',     True),
    ('MOCIÓN DE EMPLAZAMIENTO SOLICITADA POR EL DIP. FERRARO',                    True),
    ('PLAN DE LABOR',                                                             True),
    ('OD 86 - 89 - 90 -92 - 103',                                                True),
    ('Exp. 3-PE-09',                                                              True),
    ('Expediente 4259-D-2019',                                                    True),
    ('Exp. 2681-D-06 - Orden del Día 798',                                        True),
    ('Moción de Orden solicitada por el Diputado CARMONA, Guillermo',             True),
    ('Pedido de Preferencia solicitado por el Diputado PITROLA, Néstor',          True),
    ('Exp.100-S-06 - Orden del Día 567',                                          True),
    ('Exp. 2210-D-05 y otros - Orden del Día 713',                                True),
    ('Exps. 2975-D-07 y 3044-D-07',                                               True),
    ('Exp. 273-D-09 y otros - Vot. en General y Particular',                      True),
    ('Moción del Diputado Montiel de tratamiento sobre Tabla del Exp. 3931-D-95', True),
    ('Moción de vuelta a Comisión del Expediente 88-S-16, O.D. 750',              True),
    ('Exped. 2248-D-07 - Orden del día 3039',                                     True),
    ('Exp.2310-D-05 -- Orden del Día 535 - Vot. en General',                      True),
    ('Moción de la Diputada Troyano de tratamiento sobre Tabla del Exp.',         True),
    ('O.D. 492',                                                                  True),
    ('O.D. 307 - O.D. 522 - O.D. 523 - O.D. 524',                                True),
    ('Solicitud de Licencia del Sr. Diputado SABBATELLA, Martín',                 True),
    ('Solicitud de Licencia si goce de dieta, por razones particulares',          True),
    ('Habilitación de temas para la que fue convocada la sesión',                 True),
    ('VOTACION DESARROLLO DE LA SESION',                                          True),
    ('Proceder a la Apertura del sobre que contiene la Declaración Jurada',       True),
    ('Renuncia del Dip. AMERI',                                                   True),
    ('Exp.494-D-06 (y otro) - O. del Día 884',                                    True),
    ('Expte. 0089-S-2020. DE LEY. CONVENIO DE TRANSFERENCIA PROGRESIVA A CABA',  False),
    ('Exp. 498-D-21 - Ayuda Transportistas Escolares',                            False),
    ('Exp. 31-PE-09 - Democratización de la Representación Política',             False),
    ('Exp. 517-D-08 - O.D. 2148 - Programa Nacional de Asistencia a las Adicciones', False),
    ('O.D. 923 - EMERGENCIA SANITARIA DE LA SALUD PEDIÁTRICA',                    False),
    ('O.D. 721 - LEY DE FICHA LIMPIA.',                                           False),
]

errores = 0
for titulo, esperado in casos_prueba:
    resultado = es_sin_valor(titulo)
    estado = '✓' if resultado == esperado else '✗ ERROR'
    if resultado != esperado:
        errores += 1
    print(f'{estado}  filtrar={resultado}  |  {titulo[:75]}')

print(f'\n{len(casos_prueba) - errores}/{len(casos_prueba)} casos correctos.')

## ⚠️ CELDA DE INSPECCIÓN — Revisar antes de filtrar

Muestra la lista completa de títulos que se eliminarían. **No modifica nada.**
Revisá que los resultados sean razonables antes de ejecutar la celda de aplicación.

In [ ]:
# Lista de títulos únicos que serían descartados
titulos_unicos = df['titulo_base'].dropna().unique()
titulos_a_filtrar = sorted([t for t in titulos_unicos if es_sin_valor(t)])

# Conteo de filas por título
conteo = df[df['titulo_base'].isin(titulos_a_filtrar)].groupby('titulo_base').size().rename('filas')

print(f'Títulos únicos en el dataset:       {len(titulos_unicos)}')
print(f'Títulos que se filtrarían:          {len(titulos_a_filtrar)}')
print(f'Filas que se eliminarían:           {conteo.sum()}')
print()

# Verificar que los 5 borderline CON DESCRIPCIÓN no están en la lista de filtrado.
# (El sexto caso borderline, Exp. 7835-D-00 - pliegos de magistrados, SÍ se filtra
# intencionalmente porque son designaciones de personas, no leyes temáticas.)
borderline_conservar = [
    'Expte. 0089-S-2020. DE LEY. CONVENIO DE TRANSFERENCIA PROGRESIVA A CABA',
    'Expte. 0366-D-2020. DE LEY. PROMOCION DEL USO DE CISTERNAS DE DOBLE DESCARGA EN INSTALACIONES SANITARIAS',
    'Exp. 498-D-21 - Ayuda Transportistas Escolares',
    'Exp. 581-D-2009 - Sistema de Faros Centenarios',
    'Exp. 2183-D-09 -  Impuestos Internos',
]
problemas = [t for t in borderline_conservar if t in titulos_a_filtrar]
if problemas:
    print('⚠️  PROBLEMA: estos títulos están siendo filtrados incorrectamente:')
    for t in problemas:
        print(f'   - {t}')
else:
    print('✓ Los 5 títulos borderline con descripción NO están en la lista de filtrado.')

print('✓ Exp. 7835-D-00 (pliegos magistrados) SÍ se filtra (decisión de diseño).')
print()

# Mostrar lista completa con conteo de filas
print('─' * 80)
print(f'{"TÍTULO":<65} {"FILAS":>5}')
print('─' * 80)
for titulo in titulos_a_filtrar:
    n = conteo.get(titulo, 0)
    print(f'{titulo[:65]:<65} {n:>5}')

## ⚠️ CELDA DE APLICACIÓN — Ejecutar solo después de revisar la inspección

Aplica el filtro y guarda `df_modelado.csv`. No modifica `df_consolidado.csv`.

In [ ]:
filas_antes = len(df)
titulos_antes = df['titulo_base'].nunique()

# Aplicar filtro: conservar solo filas cuyo titulo_base NO está en la lista a filtrar
df_modelado = df[~df['titulo_base'].isin(titulos_a_filtrar)].copy()

filas_despues = len(df_modelado)
titulos_despues = df_modelado['titulo_base'].nunique()

print('─' * 50)
print('RESUMEN DEL FILTRADO')
print('─' * 50)
print(f'Títulos únicos eliminados:  {titulos_antes - titulos_despues}')
print(f'Filas eliminadas:           {filas_antes - filas_despues}')
print(f'Filas restantes:            {filas_despues}')
print(f'Títulos únicos restantes:   {titulos_despues}')
print()

# Guardar — nombre distinto para no sobreescribir el original
ruta_salida = '../data/df_modelado.csv'
df_modelado.to_csv(ruta_salida, index=False)
print(f'Guardado en: {ruta_salida}')

# Verificar que df_consolidado no fue modificado
df_original = pd.read_csv('../data/df_consolidado.csv')
assert len(df_original) == filas_antes, '⚠️ ERROR: df_consolidado.csv fue modificado'
print('✓ df_consolidado.csv intacto (misma cantidad de filas que al inicio).')

---
## Busqueda de autor por proyecto

Para cada titulo unico en `df_modelado`, se intenta identificar el autor del proyecto
parlamentario correspondiente usando dos estrategias en cascada:
1. **Match deterministico**: por numero de expediente extraido del titulo.
2. **Match fuzzy**: similitud TF-IDF + coseno contra el catalogo de proyectos.

> Requiere `data/proyectos_parlamentarios2.1.csv`. Sin ese archivo esta seccion falla.

In [ ]:
# ============================================================
# PARAMETROS
# ============================================================
URL_PROYECTOS   = '../data/proyectos_parlamentarios2.1.csv'
THRESHOLD_FUZZY = 0.70
BATCH_SIZE      = 200

import unicodedata
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Parametros cargados.')
print(f'  Archivo proyectos: {URL_PROYECTOS}')
print(f'  Umbral fuzzy:      {THRESHOLD_FUZZY}')
print(f'  Batch size:        {BATCH_SIZE}')

In [ ]:
# ============================================================
# PASO A — Cargar catalogo de proyectos parlamentarios
# ============================================================
df_proyectos = pd.read_csv(URL_PROYECTOS)
cols_lookup  = ['TITULO', 'EXP_DIPUTADOS', 'EXP_SENADO', 'AUTOR', 'CAMARA_ORIGEN']
df_proy      = df_proyectos[cols_lookup].copy()
print(f'[A] Proyectos cargados: {len(df_proy):,}')
print(f'    Autores unicos: {df_proy["AUTOR"].nunique():,}')
df_proy.head(3)

In [ ]:
# ============================================================
# PASO B — Tabla de trabajo sobre titulos unicos de df_modelado
# ============================================================
titulos_unicos = (
    df_modelado[['titulo_base']]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)
titulos_unicos['autor_final']    = None
titulos_unicos['camara_origen']  = None
titulos_unicos['fuente_autor']   = None
titulos_unicos['score_fuzzy']    = np.nan
print(f'[B] Titulos unicos en df_modelado: {len(titulos_unicos):,}')

# ============================================================
# PASO C — Funciones de normalizacion de expedientes
# ============================================================
def normalizar_exp(exp):
    if pd.isna(exp):
        return None
    m = re.match(r'^(\d+)-([A-Z]+)-(\d{2,4})$', str(exp).strip().upper())
    if not m:
        return str(exp).strip().upper()
    num, tipo, anio = m.groups()
    if len(anio) == 2:
        anio = f'19{anio}' if int(anio) >= 90 else f'20{anio}'
    return f'{num}-{tipo}-{anio}'

def extraer_exp_de_titulo(titulo):
    for pat in [
        r'(\d{1,5}-D-\d{2,4})',
        r'(\d{1,5}-S-\d{2,4})',
        r'(\d{1,5}-PE-\d{2,4})',
        r'(\d{1,5}-JGM-\d{2,4})',
        r'(\d{1,5}-CD-\d{2,4})',
    ]:
        m = re.search(pat, str(titulo), re.IGNORECASE)
        if m:
            return normalizar_exp(m.group(1))
    return None

# ============================================================
# PASO D — Match deterministico por expediente
# ============================================================
lookup_exp = {}
for _, row in df_proy.iterrows():
    for col in ['EXP_DIPUTADOS', 'EXP_SENADO']:
        exp = normalizar_exp(row[col])
        if exp and exp not in lookup_exp:
            lookup_exp[exp] = (row['AUTOR'], row['CAMARA_ORIGEN'])

titulos_unicos['exp_extraido'] = titulos_unicos['titulo_base'].apply(extraer_exp_de_titulo)

for idx in titulos_unicos[titulos_unicos['exp_extraido'].notna()].index:
    exp = titulos_unicos.at[idx, 'exp_extraido']
    if exp in lookup_exp:
        titulos_unicos.at[idx, 'autor_final']   = lookup_exp[exp][0]
        titulos_unicos.at[idx, 'camara_origen'] = lookup_exp[exp][1]
        titulos_unicos.at[idx, 'fuente_autor']  = 'deterministico'

n_det = (titulos_unicos['fuente_autor'] == 'deterministico').sum()
print(f'[D] Match deterministico: {n_det}/{len(titulos_unicos)} titulos')

In [ ]:
# ============================================================
# PASO E — Normalizacion de texto para TF-IDF
# ============================================================
def normalize_texto(text):
    if pd.isna(text):
        return ''
    text = str(text).upper()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r'\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}', '', text)
    text = re.sub(r'\d{1,5}-[A-Z]+-\d{2,4}', '', text)
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# ============================================================
# PASO F — Match fuzzy TF-IDF para los sin autor
# ============================================================
df_fuzzy_input = titulos_unicos[titulos_unicos['fuente_autor'].isna()].copy()
print(f'[F] Titulos para fuzzy: {len(df_fuzzy_input)}')

proy_norm    = df_proy['TITULO'].apply(normalize_texto).tolist()
query_norm   = df_fuzzy_input['titulo_base'].apply(normalize_texto).tolist()

vectorizer   = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
tfidf        = vectorizer.fit_transform(proy_norm + query_norm)
proy_matrix  = tfidf[:len(proy_norm)]
query_matrix = tfidf[len(proy_norm):]

resultados_fuzzy = []
for i in range(0, len(df_fuzzy_input), BATCH_SIZE):
    batch        = query_matrix[i:i + BATCH_SIZE]
    scores       = cosine_similarity(batch, proy_matrix)
    best_indices = np.argmax(scores, axis=1)
    best_scores  = scores[np.arange(len(best_indices)), best_indices]
    for j, (best_idx, best_score) in enumerate(zip(best_indices, best_scores)):
        resultados_fuzzy.append({
            'idx_titulos':   df_fuzzy_input.index[i + j],
            'autor_final':   df_proy.iloc[best_idx]['AUTOR']          if best_score >= THRESHOLD_FUZZY else None,
            'camara_origen': df_proy.iloc[best_idx]['CAMARA_ORIGEN']  if best_score >= THRESHOLD_FUZZY else None,
            'fuente_autor':  'fuzzy'                                   if best_score >= THRESHOLD_FUZZY else None,
            'score_fuzzy':   round(float(best_score), 3),
        })
    print(f'  Lote {i//BATCH_SIZE + 1}: procesados {min(i+BATCH_SIZE, len(df_fuzzy_input))}/{len(df_fuzzy_input)}')

for r in resultados_fuzzy:
    idx = r['idx_titulos']
    titulos_unicos.at[idx, 'autor_final']   = r['autor_final']
    titulos_unicos.at[idx, 'camara_origen'] = r['camara_origen']
    titulos_unicos.at[idx, 'fuente_autor']  = r['fuente_autor']
    titulos_unicos.at[idx, 'score_fuzzy']   = r['score_fuzzy']

# ============================================================
# REPORTE DE COBERTURA
# ============================================================
print()
print('=== COBERTURA DE AUTOR ===')
for fuente in ['deterministico', 'fuzzy', None]:
    if fuente:
        mask  = titulos_unicos['fuente_autor'] == fuente
    else:
        mask  = titulos_unicos['fuente_autor'].isna()
    label = fuente if fuente else 'sin_autor'
    print(f'  {label:<20}: {mask.sum():>5} titulos')
print(f'  TOTAL: {len(titulos_unicos):>5} titulos')

In [ ]:
# ============================================================
# PASO G — Merge de columnas de autor en df_modelado
# ============================================================
cols_autor = ['titulo_base', 'autor_final', 'camara_origen', 'fuente_autor', 'score_fuzzy']

# Descartar columnas de autor si ya existieran de una corrida anterior
cols_a_descartar = [c for c in ['autor_final', 'camara_origen', 'fuente_autor', 'score_fuzzy']
                    if c in df_modelado.columns]
if cols_a_descartar:
    df_modelado = df_modelado.drop(columns=cols_a_descartar)
    print(f'Columnas de autor previas descartadas: {cols_a_descartar}')

df_modelado = df_modelado.merge(
    titulos_unicos[cols_autor],
    on='titulo_base',
    how='left'
)

print(f'df_modelado shape: {df_modelado.shape}')
print(f'Columnas: {df_modelado.columns.tolist()}')
print(f'Filas con autor:    {df_modelado["autor_final"].notna().sum():,}')
print(f'Filas sin autor:    {df_modelado["autor_final"].isna().sum():,}')

In [ ]:
# ============================================================
# EXPORTAR df_modelado.csv con columnas de autor
# (sobreescribe la version sin autor guardada en la celda de filtrado)
# ============================================================
ruta_salida = '../data/df_modelado.csv'
df_modelado.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

print(f'Guardado: {len(df_modelado):,} filas en {ruta_salida}')
print(f'Columnas: {df_modelado.columns.tolist()}')

In [ ]:
# ============================================================
# EXPORTAR titulos_autor.xlsx — tabla de auditoria para completado manual
# Una fila por titulo unico de votacion.
# Ordenada: primero los sin autor (para facilitar el completado a mano).
# ============================================================

# Traer fecha_votacion e id_votacion del df_modelado (mas reciente por titulo)
cols_meta = ['titulo_base', 'fecha_votacion']
if 'id_votacion' in df_modelado.columns:
    cols_meta.append('id_votacion')

meta = (
    df_modelado[cols_meta]
    .sort_values('fecha_votacion', ascending=False)
    .drop_duplicates(subset='titulo_base')
    .reset_index(drop=True)
)

df_titulos_autor = (
    titulos_unicos[['titulo_base', 'autor_final', 'camara_origen', 'fuente_autor', 'score_fuzzy']]
    .merge(meta, on='titulo_base', how='left')
)

# Renombrar y reordenar columnas segun spec
rename_map = {'titulo_base': 'titulo_votacion'}
df_titulos_autor = df_titulos_autor.rename(columns=rename_map)

cols_orden = ['titulo_votacion', 'fecha_votacion']
if 'id_votacion' in df_titulos_autor.columns:
    cols_orden.append('id_votacion')
cols_orden += ['autor_final', 'fuente_autor', 'score_fuzzy']
df_titulos_autor = df_titulos_autor[cols_orden]

# Ordenar: primero sin autor (NaN en fuente_autor), luego fuzzy, luego deterministico
orden_fuente = {'deterministico': 2, 'fuzzy': 1}
df_titulos_autor['_orden'] = df_titulos_autor['fuente_autor'].map(orden_fuente).fillna(0)
df_titulos_autor = df_titulos_autor.sort_values('_orden').drop(columns='_orden').reset_index(drop=True)

df_titulos_autor.to_excel('../data/titulos_autor.xlsx', index=False)

print(f'Exportado: {len(df_titulos_autor):,} filas en ../data/titulos_autor.xlsx')
print(f'Columnas: {df_titulos_autor.columns.tolist()}')
print()
print('Distribucion por fuente_autor:')
print(df_titulos_autor['fuente_autor'].value_counts(dropna=False).to_string())